# GPU-Accelerated Canadian Fire Weather Index (FWI)
**Goal:** compute the full FWI System (FFMC, DMC, DC, ISI, BUI, FWI) for every pixel of a large weather grid on an NVIDIA GPU with **Numba CUDA**, check it against a CPU version, and measure the speedup.

**Main idea:** each day depends on the day before, so time has to run in order. Pixels don't depend on each other, though, so **one GPU thread handles one pixel** and steps through the days, keeping its moisture codes in fast registers.

**Before you run:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
import os, math, time
import numpy as np
import matplotlib.pyplot as plt
from numba import njit, prange, cuda
print('CUDA available:', cuda.is_available())
SIM = os.environ.get('NUMBA_ENABLE_CUDASIM') == '1'   # True only when testing without a GPU

## 1. The FWI equations (Van Wagner 1987)
These are plain scalar Python functions. They handle one pixel on one day. Numba then compiles the **same code** two ways: for the CPU (`njit`) and as GPU device functions (`cuda.jit(device=True)`).

In [ ]:
import math

# Day-length factors (Van Wagner 1987), index = month-1
EL = (6.5, 7.5, 9.0, 12.8, 13.9, 13.9, 12.4, 10.9, 9.4, 8.0, 7.0, 6.0)
FL = (-1.6, -1.6, -1.6, 0.9, 3.8, 5.8, 6.4, 5.0, 2.4, 0.4, -1.6, -1.6)


def ffmc_step(ffmc0, T, H, W, rain):
    mo = 147.2 * (101.0 - ffmc0) / (59.5 + ffmc0)
    if rain > 0.5:
        rf = rain - 0.5
        if mo > 150.0:
            mo = (mo + 42.5 * rf * math.exp(-100.0 / (251.0 - mo)) * (1.0 - math.exp(-6.93 / rf))
                  + 0.0015 * (mo - 150.0) ** 2 * math.sqrt(rf))
        else:
            mo = mo + 42.5 * rf * math.exp(-100.0 / (251.0 - mo)) * (1.0 - math.exp(-6.93 / rf))
        if mo > 250.0:
            mo = 250.0
    ed = 0.942 * H ** 0.679 + 11.0 * math.exp((H - 100.0) / 10.0) + 0.18 * (21.1 - T) * (1.0 - math.exp(-0.115 * H))
    if mo > ed:
        ko = 0.424 * (1.0 - (H / 100.0) ** 1.7) + 0.0694 * math.sqrt(W) * (1.0 - (H / 100.0) ** 8)
        kd = ko * 0.581 * math.exp(0.0365 * T)
        m = ed + (mo - ed) * 10.0 ** (-kd)
    else:
        ew = 0.618 * H ** 0.753 + 10.0 * math.exp((H - 100.0) / 10.0) + 0.18 * (21.1 - T) * (1.0 - math.exp(-0.115 * H))
        if mo < ew:
            k1 = 0.424 * (1.0 - ((100.0 - H) / 100.0) ** 1.7) + 0.0694 * math.sqrt(W) * (1.0 - ((100.0 - H) / 100.0) ** 8)
            kw = k1 * 0.581 * math.exp(0.0365 * T)
            m = ew - (ew - mo) * 10.0 ** (-kw)
        else:
            m = mo
    f = 59.5 * (250.0 - m) / (147.2 + m)
    return min(max(f, 0.0), 101.0)


def dmc_step(dmc0, T, H, rain, el):
    t = max(T, -1.1)
    rk = 1.894 * (t + 1.1) * (100.0 - H) * el * 1e-4
    if rain > 1.5:
        rw = 0.92 * rain - 1.27
        wmi = 20.0 + 280.0 / math.exp(0.023 * dmc0)
        if dmc0 <= 33.0:
            b = 100.0 / (0.5 + 0.3 * dmc0)
        elif dmc0 <= 65.0:
            b = 14.0 - 1.3 * math.log(dmc0)
        else:
            b = 6.2 * math.log(dmc0) - 17.2
        wmr = wmi + 1000.0 * rw / (48.77 + b * rw)
        pr = 43.43 * (5.6348 - math.log(wmr - 20.0))
    else:
        pr = dmc0
    pr = max(pr, 0.0)
    return max(pr + rk, 0.0)


def dc_step(dc0, T, rain, fl):
    t = max(T, -2.8)
    pe = max((0.36 * (t + 2.8) + fl) / 2.0, 0.0)
    if rain > 2.8:
        rd = 0.83 * rain - 1.27
        qo = 800.0 * math.exp(-dc0 / 400.0)
        qr = qo + 3.937 * rd
        dr = max(400.0 * math.log(800.0 / qr), 0.0)
        return dr + pe
    return dc0 + pe


def isi_calc(ffmc, W):
    mo = 147.2 * (101.0 - ffmc) / (59.5 + ffmc)
    ff = 91.9 * math.exp(-0.1386 * mo) * (1.0 + mo ** 5.31 / 4.93e7)
    return 0.208 * math.exp(0.05039 * W) * ff


def bui_calc(dmc, dc):
    if dmc == 0.0 and dc == 0.0:
        return 0.0
    if dmc <= 0.4 * dc:
        b = 0.8 * dmc * dc / (dmc + 0.4 * dc)
    else:
        b = dmc - (1.0 - 0.8 * dc / (dmc + 0.4 * dc)) * (0.92 + (0.0114 * dmc) ** 1.7)
    return max(b, 0.0)


def fwi_calc(isi, bui):
    if bui <= 80.0:
        bb = 0.1 * isi * (0.626 * bui ** 0.809 + 2.0)
    else:
        bb = 0.1 * isi * (1000.0 / (25.0 + 108.64 * math.exp(-0.023 * bui)))
    if bb <= 1.0:
        return bb
    return math.exp(2.72 * (0.434 * math.log(bb)) ** 0.647)


### Check against the published example
Van Wagner & Pickett (1985) use this test: April, T=17°C, RH=42%, wind=25 km/h, no rain, starting from FFMC=85, DMC=6, DC=15.
The expected result is **FFMC 87.7, DMC 8.5, DC 19.0, ISI 10.9, BUI 8.5, FWI 10.1**.

In [ ]:
f = ffmc_step(85, 17, 42, 25, 0); d = dmc_step(6, 17, 42, 0, EL[3]); c = dc_step(15, 17, 0, FL[3])
i = isi_calc(f, 25); b = bui_calc(d, c)
print(f'FFMC {f:.1f}  DMC {d:.1f}  DC {c:.1f}  ISI {i:.1f}  BUI {b:.1f}  FWI {fwi_calc(i, b):.1f}')

## 2. Compile for CPU and GPU

In [ ]:
# CPU versions
ffmc_c, dmc_c, dc_c = njit(ffmc_step), njit(dmc_step), njit(dc_step)
isi_c, bui_c, fwi_c = njit(isi_calc), njit(bui_calc), njit(fwi_calc)
# GPU device versions (same source code)
ffmc_g, dmc_g, dc_g = [cuda.jit(device=True)(f) for f in (ffmc_step, dmc_step, dc_step)]
isi_g, bui_g, fwi_g = [cuda.jit(device=True)(f) for f in (isi_calc, bui_calc, fwi_calc)]

## 3. CPU baselines
- `fwi_cpu_serial`: one CPU core, compiled with Numba. This is already about 100× faster than pure Python.
- `fwi_cpu_parallel`: all CPU cores (`prange`). This is the fair, tough baseline.

In [ ]:
def _make_cpu(parallel):
    @njit(parallel=parallel)
    def run(T, H, W, R, el, fl, out):
        D, ny, nx = T.shape
        for y in prange(ny):
            for x in range(nx):
                ffmc, dmc, dc = 85.0, 6.0, 15.0          # standard start-up values
                for d in range(D):
                    t, h, w, r = T[d, y, x], H[d, y, x], W[d, y, x], R[d, y, x]
                    ffmc = ffmc_c(ffmc, t, h, w, r)
                    dmc = dmc_c(dmc, t, h, r, el)
                    dc = dc_c(dc, t, r, fl)
                    isi = isi_c(ffmc, w)
                    bui = bui_c(dmc, dc)
                    out[d, y, x] = fwi_c(isi, bui)
    return run
fwi_cpu_serial, fwi_cpu_parallel = _make_cpu(False), _make_cpu(True)

## 4. The GPU kernel
`cuda.grid(2)` gives each thread its (x, y) pixel. The thread loops over the days and keeps FFMC, DMC, and DC in registers. Its memory reads are **coalesced**: neighbouring threads (consecutive x) read neighbouring addresses.

In [ ]:
@cuda.jit
def fwi_kernel(T, H, W, R, el, fl, out):
    x, y = cuda.grid(2)
    D, ny, nx = T.shape
    if x >= nx or y >= ny:
        return
    ffmc, dmc, dc = 85.0, 6.0, 15.0
    for d in range(D):
        t = T[d, y, x]; h = H[d, y, x]; w = W[d, y, x]; r = R[d, y, x]
        ffmc = ffmc_g(ffmc, t, h, w, r)
        dmc = dmc_g(dmc, t, h, r, el)
        dc = dc_g(dc, t, r, fl)
        isi = isi_g(ffmc, w)
        bui = bui_g(dmc, dc)
        out[d, y, x] = fwi_g(isi, bui)

TPB = (32, 8)   # 256 threads per block; 32 along x keeps memory reads coalesced
def fwi_gpu(dT, dH, dW, dR, el, fl, dout):
    D, ny, nx = dT.shape
    blocks = ((nx + TPB[0] - 1) // TPB[0], (ny + TPB[1] - 1) // TPB[1])
    fwi_kernel[blocks, TPB](dT, dH, dW, dR, el, fl, dout)
    cuda.synchronize()

## 5. Synthetic weather grid
This makes realistic-looking dry-season weather: a smooth hot and dry gradient, random noise, and scattered rain. To use real data later, swap in ERA5 or IMD grids with the same shapes: `(days, ny, nx)`.

In [ ]:
def make_weather(D, ny, nx, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.linspace(0, 1, ny, dtype=np.float32), np.linspace(0, 1, nx, dtype=np.float32), indexing='ij')
    base_T = 24 + 14 * xx * (1 - 0.5 * yy) + 3 * np.sin(6 * yy) * np.cos(4 * xx)
    base_H = 75 - 50 * xx * (1 - 0.4 * yy)
    T = np.empty((D, ny, nx), np.float32); H = np.empty_like(T); W = np.empty_like(T); R = np.empty_like(T)
    for d in range(D):
        T[d] = base_T + rng.normal(0, 2, (ny, nx)).astype(np.float32)
        H[d] = np.clip(base_H + rng.normal(0, 8, (ny, nx)), 5, 100)
        W[d] = np.clip(rng.gamma(2.0, 7.0, (ny, nx)), 0, 60)
        R[d] = np.where(rng.random((ny, nx)) < 0.15 * (1 - xx), rng.gamma(1.5, 5.0, (ny, nx)), 0).astype(np.float32)
    return T, H, W, R

MONTH = 4   # April, before the monsoon
EL_M, FL_M = EL[MONTH - 1], FL[MONTH - 1]

## 6. Correctness: GPU vs CPU

In [ ]:
n, D = (32, 3) if SIM else (512, 10)
T, H, W, R = make_weather(D, n, n)
out_cpu = np.empty_like(T); fwi_cpu_parallel(T, H, W, R, EL_M, FL_M, out_cpu)
out_ser = np.empty_like(T); fwi_cpu_serial(T, H, W, R, EL_M, FL_M, out_ser)   # also compiles the serial version before benchmarking
print('serial == parallel:', np.allclose(out_ser, out_cpu))
dT, dH, dW, dR = [cuda.to_device(a) for a in (T, H, W, R)]
dout = cuda.device_array_like(T); fwi_gpu(dT, dH, dW, dR, EL_M, FL_M, dout)
out_gpu = dout.copy_to_host()
print('max |GPU - CPU| =', np.abs(out_gpu - out_cpu).max())
print('FWI range:', out_cpu.min().round(2), 'to', out_cpu.max().round(2))

## 7. Benchmark
Each size runs for 10 days. The GPU is timed two ways: **kernel only** (data already on the GPU) and **end-to-end** (copy in, compute, copy out). Report both in your README so the comparison is honest.

In [ ]:
def best_time(fn, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return min(ts)

SIZES = [32, 64] if SIM else [256, 512, 1024, 2048]
D = 2 if SIM else 10
rows = []
for n in SIZES:
    T, H, W, R = make_weather(D, n, n, seed=n)
    out = np.empty_like(T)
    t_ser = best_time(lambda: fwi_cpu_serial(T, H, W, R, EL_M, FL_M, out), reps=1)
    t_par = best_time(lambda: fwi_cpu_parallel(T, H, W, R, EL_M, FL_M, out))
    dT, dH, dW, dR = [cuda.to_device(a) for a in (T, H, W, R)]
    dout = cuda.device_array_like(T)
    fwi_gpu(dT, dH, dW, dR, EL_M, FL_M, dout)              # warm-up (JIT compile)
    t_gpu = best_time(lambda: fwi_gpu(dT, dH, dW, dR, EL_M, FL_M, dout))
    def e2e():
        a = [cuda.to_device(x) for x in (T, H, W, R)]; o = cuda.device_array_like(T)
        fwi_gpu(*a, EL_M, FL_M, o); o.copy_to_host()
    t_e2e = best_time(e2e)
    rows.append((n, t_ser, t_par, t_gpu, t_e2e))
    print(f'{n:>5}² x {D}d | CPU 1-core {t_ser*1e3:9.1f} ms | CPU all-cores {t_par*1e3:8.1f} ms | '
          f'GPU kernel {t_gpu*1e3:7.2f} ms | GPU end-to-end {t_e2e*1e3:8.1f} ms | '
          f'speedup vs all-cores: {t_par/t_gpu:6.1f}x kernel, {t_par/t_e2e:5.1f}x e2e')

In [ ]:
r = np.array(rows)
px = r[:, 0] ** 2 * D / 1e6
plt.figure(figsize=(8, 5))
for col, lab in [(1, 'CPU 1 core'), (2, 'CPU all cores'), (4, 'GPU end-to-end'), (3, 'GPU kernel only')]:
    plt.plot(px, px / r[:, col], 'o-', label=lab)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Pixel-days (millions)'); plt.ylabel('Throughput (million pixel-days / s)')
plt.title('FWI System throughput: CPU vs GPU (Numba CUDA)'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('fwi_benchmark.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. India-scale demo (~1 km grid)
India's land area is about 3.3 million km², so a 3000 × 3000 grid at 1 km covers it. This runs a full week of FWI for **about 9 million pixels**.

In [ ]:
ny = nx = 64 if SIM else 3000
D = 2 if SIM else 7
T, H, W, R = make_weather(D, ny, nx, seed=42)
t0 = time.perf_counter()
dT, dH, dW, dR = [cuda.to_device(a) for a in (T, H, W, R)]
dout = cuda.device_array_like(T); fwi_gpu(dT, dH, dW, dR, EL_M, FL_M, dout)
fwi_week = dout.copy_to_host()
print(f'{ny*nx/1e6:.1f} M pixels x {D} days in {time.perf_counter()-t0:.2f} s (incl. transfers)')

# Canadian danger classes: Low <5.2, Moderate <11.2, High <21.3, Very High <38, Extreme >=38
bounds = [0, 5.2, 11.2, 21.3, 38, 1e9]
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap(['#2b83ba', '#abdda4', '#ffffbf', '#fdae61', '#d7191c'])
plt.figure(figsize=(7, 6))
im = plt.imshow(fwi_week[-1], cmap=cmap, norm=BoundaryNorm(bounds, cmap.N), origin='lower')
cb = plt.colorbar(im, ticks=[2.6, 8.2, 16, 30, 60]); cb.ax.set_yticklabels(['Low', 'Moderate', 'High', 'Very high', 'Extreme'])
plt.title(f'Synthetic FWI, day {D}'); plt.axis('off')
plt.savefig('fwi_map.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Next steps (each makes a good commit)
1. **float32 math:** the T4 runs FP32 about 32× faster than FP64. Cast the constants with `np.float32(...)` and use `fastmath=True`, then measure the gain and check the error stays small.
2. **Profile:** run `!ncu` (Nsight Compute) or `numba.cuda.profile_start()` and find out whether the kernel is limited by memory or by compute.
3. **Block-size sweep:** try (16,16), (32,4), (32,16), (64,4) and plot the results.
4. **Only store the last day:** write FWI only for the final day to cut memory traffic.
5. **Real data:** load ERA5 or IMD gridded temperature, humidity, wind, and rain with `xarray`, run them through the same kernel, and compare with your Tempest station values.
6. **CuPy version:** write the same thing as a `cp.ElementwiseKernel` or `RawKernel` and compare.

**Resume line:** *"GPU-accelerated the Canadian FWI System with Numba CUDA (one thread per pixel, register-resident state). Validated against Van Wagner reference values, and X× faster than multicore CPU on a 9M-pixel India-scale grid (T4)."*